<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;"><div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div><div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div></div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Image Segmentation — Algorithm</b></h1></div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Hand, tower, and peppers images |
| **Final outputs** | Binary/region segmentations, region measurements, evaluation metrics, method-selection diagnostics, 21 required figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project kernel and use this notebook as the coding plan for `main.ipynb`.
2. Adopt one consistent foreground-mask convention across all methods.
3. Keep controlled metric demonstrations distinct from external ground truth.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Setup, Paths, and Image Roles

**Purpose**

Assign fixed roles to input images.

**Inputs**

Hand, tower and peppers files.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, PIL, SciPy `ndimage`, and OpenCV.
2. Locate the lab root and create data/output paths.
3. Discover supported images and map hand, tower, and peppers roles deterministically.
4. Create reusable grayscale/RGB loading and figure-saving helpers.

**Outputs**

Paths, loaders and figure helper.

**Validation**

Each role resolves; grayscale/RGB conversions are explicit.

**On failure**

Stop if a role is missing rather than replacing the scene.

## 2. Define Mask Conversion and Overlay Helpers

**Purpose**

Separate logical masks from image encodings.

**Inputs**

Boolean mask and same-sized RGB image.

**Procedure**

1. Keep logical masks as boolean arrays inside the notebook.
2. Convert to uint8 0/255 only at OpenCV boundaries.
3. Create an overlay helper that highlights foreground on RGB images.
4. Verify mask and image spatial dimensions match.

**Outputs**

0/255 conversion and overlay functions.

**Validation**

Mask spatial shape matches image; conversion preserves foreground.

**On failure**

Correct shape/channel mismatches before overlaying.

## 3. Define the Thresholding Convention

**Purpose**

Keep threshold polarity consistent.

**Inputs**

2-D grayscale image, threshold and polarity.

**Procedure**

1. Create a helper accepting grayscale image, threshold, and foreground polarity.
2. For dark foreground return `image<threshold`; for bright foreground return `image>=threshold`.
3. Reject unsupported polarity modes.
4. Reuse this helper for manual-threshold baselines.

**Outputs**

Boolean threshold mask.

**Validation**

Dark means value<threshold; bright means value>=threshold.

**On failure**

Existing helper rejects unsupported polarity and non-2-D input.

## 4. Load the Three Lab Images

**Purpose**

Confirm the experiment inputs.

**Inputs**

Resolved image paths.

**Procedure**

1. Load the hand as grayscale and tower/peppers as RGB.
2. Verify dimensions, dtypes, and non-empty arrays.
3. Display or save one common source-image overview.

**Outputs**

Hand grayscale; tower/peppers RGB; source overview.

**Validation**

uint8 nonempty arrays with appropriate dimensions.

**On failure**

Stop on decoding errors or wrong image roles.

## 5. Inspect the Hand Histogram

**Purpose**

Support the manual threshold with intensity evidence.

**Inputs**

8-bit hand image.

**Procedure**

1. Flatten the hand image and compute the full 8-bit histogram.
2. Use the same intensity range as later global thresholds.
3. Display the histogram as evidence for the initial manual threshold.

**Outputs**

256-bin histogram.

**Validation**

Total count equals image pixels; same intensity range as thresholding.

**On failure**

Correct histogram range before choosing the threshold.

## 6. Create a Manual Global-Threshold Baseline

**Purpose**

Establish a simple interpretable baseline.

**Inputs**

Hand image and declared threshold.

**Procedure**

1. Choose the declared manual threshold.
2. Create a dark-foreground boolean mask.
3. Display original, mask, and overlay.
4. Save this baseline before automatic methods.

**Outputs**

Dark-foreground mask and overlay.

**Validation**

Boolean mask preserves spatial shape; inspect hand/background polarity.

**On failure**

Correct inversion before comparing automatic methods.

## 7. Test Threshold Sensitivity

**Purpose**

Expose instability around the manual choice.

**Inputs**

Same image and nearby thresholds.

**Procedure**

1. Choose several thresholds around the baseline.
2. Apply identical foreground semantics to all.
3. Display masks with identical layout.
4. Use the comparison to reveal unstable regions.

**Outputs**

Threshold-sensitivity grid.

**Validation**

Only threshold varies; foreground convention unchanged.

**On failure**

Do not interpret polarity changes as threshold sensitivity.

## 8. Apply Otsu Thresholding

**Purpose**

Choose a global threshold from the histogram.

**Inputs**

Hand grayscale array.

**Procedure**

1. Call OpenCV thresholding with `THRESH_BINARY_INV + THRESH_OTSU` because the hand is darker foreground.
2. Record the returned Otsu threshold.
3. Convert the OpenCV mask to boolean.
4. Display and save the result.

**Outputs**

Otsu threshold and boolean mask.

**Validation**

Use inverted binary output for dark hand foreground.

**On failure**

Inspect a poor mask as a limitation of global separation, not a failed function call.

## 9. Smooth Before Otsu

**Purpose**

Test noise reduction before threshold estimation.

**Inputs**

Hand image and fixed Gaussian kernel.

**Procedure**

1. Apply a small Gaussian blur to the hand image.
2. Recompute Otsu on the smoothed image.
3. Compare threshold and mask against unsmoothed Otsu.
4. Keep blur kernel fixed and explicit.

**Outputs**

Smoothed image, new Otsu threshold and mask.

**Validation**

Compare against unsmoothed Otsu on the same source.

**On failure**

Report erased thin details; blur is not automatically beneficial.

## 10. Compare Adaptive Thresholds

**Purpose**

Compare local threshold estimators.

**Inputs**

Same source, odd block size>1 and offset.

**Procedure**

1. Apply adaptive mean and adaptive Gaussian thresholding.
2. Use the same block size and offset for a fair comparison.
3. Use inverted binary output to preserve foreground semantics.
4. Compare both masks with global methods.

**Outputs**

Adaptive mean/Gaussian masks.

**Validation**

Same block/offset and inverted polarity for both.

**On failure**

Correct invalid block settings; inspect texture-induced foreground.

## 11. Implement a Morphology Helper

**Purpose**

Reuse morphological operations with one encoding.

**Inputs**

Mask, operation, kernel and iterations.

**Procedure**

1. Accept mask, operation name, structuring element, and iteration count.
2. Convert boolean mask to uint8 once at the OpenCV boundary.
3. Route `erode`, `dilate`, `open`, and `close` correctly.
4. Return a boolean mask and reject invalid operations or iteration counts.

**Outputs**

Boolean result mask.

**Validation**

Supported operations erode/dilate/open/close; iterations>=1.

**On failure**

Existing helper raises for unknown operation or invalid iterations.

## 12. Create Structuring Elements

**Purpose**

Make neighborhood shape explicit.

**Inputs**

Common nominal kernel size.

**Procedure**

1. Build rectangular, elliptical, and cross kernels with the same nominal size.
2. Visualize their supports.
3. Use this to make shape assumptions explicit before morphology.

**Outputs**

Rectangular, ellipse and cross supports.

**Validation**

Supports are nonempty and share declared dimensions.

**On failure**

Correct kernel shape before interpreting topology changes.

## 13. Apply Erosion and Dilation

**Purpose**

Measure foreground shrinkage and growth.

**Inputs**

Working mask and same structuring element.

**Procedure**

1. Convert the working mask to uint8.
2. Apply one erosion and one dilation with the chosen kernel.
3. Convert outputs back to boolean.
4. Compare how foreground extent changes.

**Outputs**

Eroded and dilated masks.

**Validation**

Same source, kernel and iteration count; foreground convention fixed.

**On failure**

Check inverted polarity if the interpretation appears reversed.

## 14. Apply Opening and Closing

**Purpose**

Compare removal of small foreground with closure of gaps.

**Inputs**

Same starting mask and kernel.

**Procedure**

1. Use opening to remove small foreground structures.
2. Use closing to fill small gaps/connect nearby foreground structures.
3. Apply both to the same starting mask with the same kernel.
4. Save the comparison.

**Outputs**

Opening/closing comparison.

**Validation**

Do not feed one comparison branch into the other.

**On failure**

Reduce kernel size if valid structures disappear or merge excessively.

## 15. Compute the Morphological Gradient

**Purpose**

Expose the morphological boundary band.

**Inputs**

Binary mask and chosen kernel.

**Procedure**

1. Apply OpenCV morphological gradient to the binary mask.
2. Interpret the result as a boundary-focused representation.
3. Display mask and gradient together.

**Outputs**

Morphological gradient.

**Validation**

Compute dilation-minus-erosion through OpenCV; preserve dimensions.

**On failure**

Treat this as a boundary representation, not a region label map.

## 16. Fill Holes

**Purpose**

Fill enclosed background regions.

**Inputs**

Working boolean mask.

**Procedure**

1. Convert the working mask to boolean.
2. Apply SciPy `binary_fill_holes`.
3. Compare before/after.
4. Pass only the filled boolean mask to connected-component analysis.

**Outputs**

Hole-filled boolean mask.

**Validation**

Same shape; existing foreground retained.

**On failure**

Inspect topology: filling real holes may be inappropriate for another object.

## 17. Label Connected Components

**Purpose**

Assign identities to connected foreground regions.

**Inputs**

Filled mask encoded as uint8.

**Procedure**

1. Convert the filled mask to uint8.
2. Call `connectedComponentsWithStats` with 8-connectivity.
3. Treat label 0 as background.
4. Extract foreground component areas from the returned statistics.

**Outputs**

Labels, statistics, centroids and component count.

**Validation**

8-connectivity; label 0 is background; stats align with label IDs.

**On failure**

Do not include background area among object measurements.

## 18. Remove Small Components

**Purpose**

Remove regions by a stated pixel-area rule.

**Inputs**

Component labels/stats and minimum area.

**Procedure**

1. Create an empty boolean result mask.
2. For each foreground label read its area.
3. Keep only labels whose area meets the minimum threshold.
4. Combine retained labels into the cleaned mask.

**Outputs**

Cleaned boolean mask.

**Validation**

Keep foreground labels with area>=threshold; never keep label 0.

**On failure**

If all foreground disappears, report the parameter effect rather than fabricate an object.

## 19. Extract External Contours

**Purpose**

Trace external object boundaries.

**Inputs**

Cleaned mask.

**Procedure**

1. Convert the cleaned mask to uint8.
2. Call `findContours` with external retrieval.
3. Draw all contours on an RGB copy of the hand image.
4. Save the contour overlay.

**Outputs**

External contours and RGB overlay.

**Validation**

Mask converted to uint8; retrieval excludes internal holes intentionally.

**On failure**

Handle an empty contour list; it means no detected region.

## 20. Measure Region Properties

**Purpose**

Quantify the extracted regions.

**Inputs**

Contours.

**Procedure**

1. For each contour compute area, perimeter, and bounding rectangle.
2. Compute any declared compactness/shape quantity with zero-division protection.
3. Store one structured record per region.
4. Print the region table consistently.

**Outputs**

Area, perimeter, bounding box and shape records.

**Validation**

Guard zero perimeter; contour area is not identical to pixel-count area.

**On failure**

Skip or mark undefined shape measures for degenerate contours.

## 21. Segment by Color in HSV

**Purpose**

Use hue separation with wraparound for red.

**Inputs**

RGB peppers converted to OpenCV HSV.

**Procedure**

1. Convert peppers from RGB to HSV.
2. Inspect hue, saturation, and value channels.
3. Define two red hue intervals because red wraps around the hue axis.
4. Threshold both ranges with minimum saturation/value constraints and combine masks.
5. Clean or overlay the final color mask as required.

**Outputs**

Two red-range masks and their union.

**Validation**

OpenCV 8-bit hue uses 0..179; saturation/value suppress achromatic/dark pixels.

**On failure**

Correct RGB/BGR and hue units before tuning ranges.

## 22. Create an Edge-Based Segmentation Example

**Purpose**

Build a boundary-based diagnostic.

**Inputs**

Tower grayscale, blur and Canny thresholds.

**Procedure**

1. Convert tower to grayscale.
2. Apply Gaussian smoothing.
3. Run Canny with explicit lower and upper thresholds.
4. Use morphology if needed to connect fragmented edges.
5. Save the edge-based diagnostic.

**Outputs**

Edges and morphological connection example.

**Validation**

Lower/upper thresholds ordered; source dimensions preserved.

**On failure**

Do not treat an edge map alone as filled object segmentation.

## 23. Compute the Distance Transform

**Purpose**

Measure foreground distance to background.

**Inputs**

Cleaned binary foreground mask.

**Procedure**

1. Convert the cleaned foreground mask to uint8.
2. Compute Euclidean distance to background.
3. Display distance map beside the binary mask.
4. Use high-distance interior pixels as marker evidence.

**Outputs**

Nonnegative distance map.

**Validation**

Background distance is zero; larger values indicate interior points.

**On failure**

Inspect empty/all-foreground masks before using marker thresholds.

## 24. Run Watershed Segmentation

**Purpose**

Separate regions using labelled markers.

**Inputs**

Peppers image, initial mask and distance map.

**Procedure**

1. Choose peppers as the watershed source.
2. Create an initial binary foreground estimate.
3. Use morphology to obtain sure background.
4. Threshold the distance transform for sure foreground.
5. Label foreground markers and mark unknown regions.
6. Run OpenCV watershed on a compatible color image.
7. Treat label `-1` as watershed boundary and visualize the separated regions.

**Outputs**

Watershed labels and boundary overlay.

**Validation**

Markers int32; unknown=0 before watershed; boundary=-1 afterward; image is 3-channel.

**On failure**

Inspect missing/merged markers if objects remain joined or fragment.

## 25. Implement Segmentation Metrics

**Purpose**

Define overlap metrics through explicit confusion counts.

**Inputs**

Same-shaped reference and predicted masks.

**Procedure**

1. Require reference and prediction to have the same spatial shape; review this precondition because the current helper does not explicitly check it.
2. Convert both masks to boolean and count TP, TN, FP and FN with logical operations.
3. Compute accuracy, precision, recall, IoU and Dice using the current `1e-12` denominator epsilon.
4. Explain undefined/empty-mask conventions explicitly: the implementation returns zero for zero-overlap denominators instead of assigning perfect similarity to an empty pair.

**Outputs**

TP,TN,FP,FN and five metrics.

**Validation**

Boolean masks; counts partition all pixels; epsilon=1e-12 in denominators.

**On failure**

Require equal shapes to avoid broadcasting; document empty-mask conventions rather than claim perfect overlap automatically.

## 26. Create a Controlled Metric Demonstration

**Purpose**

Demonstrate metric behavior with a controlled reference.

**Inputs**

Cleaned mask and morphological perturbation.

**Procedure**

1. Use the cleaned mask as a controlled reference.
2. Create a perturbed prediction by erosion or dilation.
3. Compute all metrics.
4. State clearly this is a metric demonstration, not external annotated ground truth.

**Outputs**

Metric example.

**Validation**

Reference is derived from this pipeline, not independent annotation.

**On failure**

Do not report these scores as real-world segmentation accuracy.

## 27. Verify the Dice–IoU Relationship

**Purpose**

Check the algebraic relation between overlap measures.

**Inputs**

IoU and Dice from the same pair.

**Procedure**

1. Read IoU and Dice from the same mask pair.
2. Compute Dice from IoU and IoU from Dice analytically.
3. Assert reconstructed values match direct measurements.

**Outputs**

Reconstructed Dice/IoU values.

**Validation**

Dice=2*IoU/(1+IoU); IoU=Dice/(2-Dice), compared with isclose.

**On failure**

Check denominator conventions and mask pairing on mismatch.

## 28. Compare Under- and Over-Segmentation

**Purpose**

Separate missed foreground from excess foreground.

**Inputs**

Same nonempty reference and erosion/dilation perturbations.

**Procedure**

1. Create an eroded version of the reference mask to simulate missing foreground.
2. Create a dilated version to simulate excess foreground.
3. Compute the same metrics for both.
4. Compare precision/recall behaviour using the unchanged reference mask.

**Outputs**

Under/over-segmentation metrics.

**Validation**

Compare precision and recall while keeping reference fixed.

**On failure**

Inspect empty masks before generalizing the expected trend.

## 29. Implement the End-to-End Dark-Foreground Pipeline

**Purpose**

Combine dark-object operations in a reusable order.

**Inputs**

2-D grayscale image, blur kernel and minimum area.

**Procedure**

1. Create a function receiving grayscale image, blur kernel, and minimum component area.
2. Gaussian-blur the image.
3. Estimate dark foreground using inverted Otsu thresholding.
4. Apply morphology, fill holes, label components, and remove small components.
5. Return threshold, intermediate masks, and final boolean mask in a structured result.

**Outputs**

Dictionary: threshold, blurred, binary, cleaned, filled, mask.

**Validation**

Otsu inverse, opening/closing, hole filling and area filtering preserve spatial shape.

**On failure**

Inspect intermediate masks to locate failure; do not tune only the final overlay.

## 30. Implement Method-Selection Logic

**Purpose**

Explain method choice from explicit scene properties.

**Inputs**

Flags for touching objects, color and illumination.

**Procedure**

1. Create a decision helper based on uneven illumination, discriminative color, and touching objects.
2. If objects touch, select distance transform plus marker-controlled watershed.
3. Else if color is discriminative, select color-space thresholding.
4. Else if illumination is uneven, select adaptive thresholding.
5. Otherwise select global/Otsu thresholding.

**Outputs**

Recommended strategy string.

**Validation**

Priority: touching objects, then color, then uneven illumination, then global threshold.

**On failure**

This helper recommends a strategy; it does not run the chosen algorithm.

## 31. Run the Integrated Segmentation Workflow

**Purpose**

Present the complete hand experiment.

**Inputs**

Hand image and dark-foreground pipeline.

**Procedure**

1. Run the reusable dark-foreground pipeline on the hand image.
2. Build the final colored overlay.
3. Display major intermediate stages and final result in one figure.
4. Save the integrated workflow output.

**Outputs**

Intermediate grid and final overlay.

**Validation**

Final overlay uses the returned mask and the same source.

**On failure**

Diagnose empty/overfilled regions through stored intermediate masks.

## 32. Run Final Validation

**Purpose**

Check masks, metric identities and figures.

**Inputs**

Computed masks, metrics and REQUIRED_OUTPUTS.

**Procedure**

1. Verify masks preserve source shape and boolean semantics where expected.
2. Verify metric values are finite and in valid ranges.
3. Verify Dice/IoU consistency checks pass.
4. Verify all declared output files exist.
5. Print the final pass message only when all checks succeed.

**Outputs**

Final validation outcome.

**Validation**

Same-shaped boolean masks; bounded finite metrics; Dice/IoU checks; 21 required PNGs.

**On failure**

Stop on actual failed assertions/missing files; passing them does not establish accuracy without external ground truth.

## Complete End-to-End Pseudocode

This is control-flow pseudocode, not an additional Python API. It follows the current implementation; review requirements and known gaps are distinguished below.

```text
resolve hand/tower/peppers inputs; define mask conversion and overlays
fix dark/bright threshold semantics; load sources and inspect the hand histogram
compare manual threshold and nearby thresholds using fixed polarity
compute inverted Otsu, Gaussian-before-Otsu and adaptive mean/Gaussian results
define boolean-mask morphology with uint8 conversion at OpenCV boundaries
compare structuring elements, erosion/dilation, opening/closing and gradient
fill holes; label 8-connected components; exclude background label 0
retain components with area >= minimum; extract external contours
measure contour area, perimeter and bounding boxes with denominator guards
convert peppers RGB to HSV; combine both red hue intervals
smooth tower grayscale; compute Canny and the edge-connection example
compute foreground distance transform
build watershed foreground/background/unknown markers on peppers
run watershed using int32 markers; draw labels and boundary -1
for each same-shaped reference/prediction pair:
    convert to boolean; count TP,TN,FP,FN
    compute accuracy, precision, recall, IoU and Dice with declared epsilon
use a pipeline-derived reference only as a controlled metric demonstration
verify Dice/IoU identities; compare erosion/dilation perturbations
implement dark-object pipeline:
    Gaussian blur -> inverted Otsu -> opening/closing -> fill holes
    label components -> area filter -> return all intermediate masks
recommend a strategy from explicit scene-property flags
run the hand pipeline; save its intermediate/final overlay grid
run existing mask/metric checks and verify all 21 required figures
```

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Segmentation Problem Formulation
- **Task 2:** Load the Lab Images
- **Task 3:** Histogram-Based Threshold Selection
- **Task 4:** Manual Global Thresholding
- **Task 5:** Threshold Sensitivity
- **Task 6:** Otsu Thresholding
- **Task 7:** Gaussian Smoothing Before Thresholding
- **Task 8:** Adaptive Thresholding
- **Task 9:** Morphological Processing
- **Task 10:** Structuring Elements
- **Task 11:** Erosion and Dilation
- **Task 12:** Opening and Closing
- **Task 13:** Morphological Gradient
- **Task 14:** Hole Filling
- **Task 15:** Connected Components
- **Task 16:** Remove Small Components
- **Task 17:** Contours
- **Task 18:** Region Properties
- **Task 19:** Color Segmentation
- **Task 20:** Edge-Based Segmentation
- **Task 21:** Distance Transform
- **Task 22:** Watershed Segmentation
- **Task 23:** Ground Truth and Segmentation Metrics
- **Task 24:** Dice vs IoU Relationship
- **Task 25:** Under-Segmentation vs Over-Segmentation
- **Task 26:** End-to-End Binary Segmentation Pipeline
- **Task 27:** Segmentation Method Selection Criteria
- **Task 28:** Integrated Segmentation Workflow


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation**.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 21 files under `../outputs/figures/`:

- `01_input_images.png`
- `02_hand_histogram.png`
- `03_manual_threshold.png`
- `04_threshold_sensitivity.png`
- `05_otsu_threshold.png`
- `06_smoothing_before_otsu.png`
- `07_adaptive_thresholding.png`
- `08_structuring_elements.png`
- `09_erosion_dilation.png`
- `10_opening_closing.png`
- `11_morphological_gradient.png`
- `12_hole_filling.png`
- `13_connected_components.png`
- `14_component_area_filtering.png`
- `15_contours.png`
- `16_hsv_channels.png`
- `17_color_segmentation.png`
- `18_edge_based_segmentation.png`
- `19_distance_transform.png`
- `20_watershed.png`
- `21_complete_pipeline.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/image-processing/segmentation/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `locate_lab_root`
- `select_input_image`
- `load_grayscale_image`
- `load_rgb_image`
- `display_grayscale_image`
- `display_rgb_image`
- `save_output_figure`
- `convert_mask_to_uint8`
- `overlay_segmentation_mask`
- `create_threshold_mask`
- `apply_morphological_operation`
- `compute_segmentation_metrics`
- `segment_dark_foreground_object`
- `recommend_segmentation_strategy`

Examples of **existing runtime assertions**, copied from the implementation:

```python
assert DATA_FILES
assert np.isclose(dice_from_iou, demo_dice)
assert np.isclose(iou_from_dice, demo_iou)
```

This excerpt is not a new test suite and does not list every runtime exception.

## Implementation Limits and Interpretation

- Mask-derived reference examples demonstrate metric behavior; they are not independently annotated ground truth.
- The current metric helper relies on callers for matching shapes and uses an epsilon convention for empty denominators. State that convention when interpreting empty masks.
- The recommendation helper returns text according to scene-property flags; it does not dispatch the selected segmentation method.
- Region and edge representations differ: contour area, component pixel count, boundary maps and watershed labels are not interchangeable metrics.

## 33. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.